In [1]:
import argparse
import tqdm
import os
import yaml
import time
import numpy as np
import pandas as pd
import sys
from collections import OrderedDict
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from torch.utils import data
import torch.backends.cudnn as cudnn
import torch.nn.functional as F

from models.model import fMRICNN

from data_loader.dataloader import fMRICNNcustomDataset

from losses.crossentropy import categorical_cross_entropy

import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

torch.Size([35, 4, 5, 6, 5])


In [2]:
parser = argparse.ArgumentParser()

# model
parser.add_argument('--epochs', default=100, type=int, metavar='N',
                    help="number of epochs to run")

parser.add_argument('-b', '--batch_size', default=8, type=int, metavar='N',
                    help="mini-batch size (default: 8)")

parser.add_argument('--early_stopping', default=40, type=int, metavar='N',
                    help="early-stopping (default: 40)")

parser.add_argument('--num_workers', default=8, type=int)

parser.add_argument('--optimizer', default='Adam', choices=['Adam', 'SGD'],
                    help="Optimizer (default: Adam)")

parser.add_argument('--lr', default=1e-4, type=float, metavar='LR',
                    help="initial learning rate")

parser.add_argument('--weight_decay', default=1e-5, type=float,
                    help="weight decay")

config = parser.parse_args(args=[])

def train(trainloader, net, criterion, optimizer, train_len, config):
	training_start_time = time.time()
	net.train()
	
	#initialization
	n_classes = 3
	best_dice = 0
	train_losses, val_losses= [], []
	running_loss = 0

	pbar = tqdm(total=(train_len/config.batch_size), desc='Training')

	for tr_idx, data_samples in (enumerate(trainloader)):
		optimizer.zero_grad()
		volume, labels = data_samples
		volume = volume.cuda()
		labels = labels.long().cuda()
		outputs = net(volume)
		print("labels únicos:", torch.unique(labels))
		print("min:", labels.min().item())
		print("max:", labels.max().item())
		loss = criterion(input_=outputs, target =labels) 

		loss.backward()
		torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
		optimizer.step()

		running_loss += loss.item()
		pbar.update(1)
	pbar.close()
	training_loss = running_loss / (2 * tr_idx)
	
	return training_loss

def validate(valloader, net, criterion, config, dataset):
	#validate
	net.eval()

	#initialization
	n_classes = 3
	val_loss = 0

	with torch.no_grad():
		pbar = tqdm(total=(len(valloader)/config.batch_size), desc='Validation')

		for val_idx, data_samples in enumerate(valloader):
			volume, labels = data_samples
			volume = volume.cuda()

			labels = labels.long().cuda()
				
			outputs = net(volume)
			validation_loss_current_model = criterion(input_=outputs, target =labels)
			val_loss += criterion(input_=outputs, target =labels)

			pbar.update(1)
		pbar.close()
	validation_loss = val_loss / (2 * val_idx)
	return validation_loss

In [3]:
import glob
import torch
from collections import Counter

contador = Counter()

BASE_OUTPUT     = "/home/al.pedro.alberti/Downloads/dataset/data"
arquivos = glob.glob(BASE_OUTPUT + "/processed/train/**/*.p", recursive=True)

for arq in arquivos[:1000]:

    sample = torch.load(arq)

    contador[int(sample["y"])] += 1

print(contador)

Counter({3: 435, 1: 227, 0: 216, 2: 122})


In [ ]:
#Make model output directory
file_name = 'cnn_{}_{}_{}'.format(config.batch_size, config.optimizer, config.lr)

BASE_OUTPUT_DIR = "/mnt/storage_C1/PedroAlberti/ml2/ml2_trabalhos_2026/trabalho_final/cnn_final/model_outputs"
output_path = os.path.join(BASE_OUTPUT_DIR, file_name)

# Cria a estrutura completa de pastas (se não existir)
os.makedirs(output_path, exist_ok=True)
print("Creating directory called {}".format(file_name))

print('-' * 20)
print("Configuration settings as follows:")
for key, val in vars(config).items():
	print('{}: {}'.format(key, val))
print('-' * 20)

#Save configuration
with open('model_outputs/{}/config.yml'.format(file_name), 'w') as f:
	yaml.dump(vars(config), f)

cudnn.benchmark = True

#Create model
print("=> Creating model")
model = fMRICNN()
model = model.cuda()

if torch.cuda.device_count() > 1:
	print("Let's use {} GPUs!".format(torch.cuda.device_count()))
	model = nn.DataParallel(model)

params = filter(lambda p: p.requires_grad, model.parameters())

if config.optimizer == 'Adam':
	optimizer = optim.Adam(params, lr=config.lr, weight_decay=config.weight_decay)
	poly_lr = lambda epoch, max_epochs=config.epochs, initial_lr=config.lr: initial_lr * (1 - epoch/max_epochs) ** 0.9
	scheduler = LambdaLR(optimizer, lr_lambda=poly_lr)
elif config.optimizer == 'SGD':
	optimizer = optim.SGD(params, lr=config.lr, weight_decay=config.weight_decay)
	poly_lr = lambda epoch, max_epochs=config.epochs, initial_lr=config.lr: initial_lr * (1 - epoch/max_epochs) ** 0.9
	scheduler = LambdaLR(optimizer, lr_lambda=poly_lr)
else:
	raise NotImplementedError

	#Dataset loading
dataset = fMRICNNcustomDataset("/home/al.pedro.alberti/Downloads/dataset/data/processed/train")

	#Split data randomly into train and test with 80% training and 20% validation
train_len=int(len(dataset)*0.8)
val_len=len(dataset)-train_len
train_dataset,val_dataset=torch.utils.data.random_split(dataset,(train_len,val_len),generator=torch.Generator().manual_seed(42))
dl_tr = data.DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=config.num_workers, pin_memory=True)
dl_val = data.DataLoader(val_dataset, batch_size=config.batch_size, shuffle=True, drop_last=True, num_workers=config.num_workers, pin_memory=True)
#metrics
criterion = categorical_cross_entropy()
log = pd.DataFrame(index=[], columns=['epoch','lr','loss','train_loss','val_loss'])
best_val_loss=100
trigger=0

fit_loop = tqdm(range(config.epochs), desc='Training')

for epoch in fit_loop:
	#train for 1 epoch
	train_log = train(dl_tr, model, criterion, optimizer, train_len, config)
	
	#evaluate on validation set
	val_log = validate(dl_val, model, criterion, config, val_dataset)

	#Update Learning Rate		
	poly_lr(epoch)
	scheduler.step()

	print('Training epoch [{}/{}], Training loss:{:.4f}, Validation loss:{:.4f}'.format(
					epoch + 1,
					config.epochs,
					train_log,
					val_log))

	tmp = pd.Series([
		epoch,
		config.lr,
		train_log,
		val_log,
	], index=['epoch','lr','loss','val_loss'])

	log = log.append(tmp, ignore_index=True)
	log.to_csv('model_outputs/{}/log.csv'.format(file_name), index=False)

	trigger += 1

	
	if val_log < best_val_loss:
		if torch.cuda.device_count() > 1:
			torch.save(model.module.state_dict(), 'model_outputs/{}/best_model.pth'.format(file_name))
		else:
			torch.save(model.state_dict(), 'model_outputs/{}/best_model.pth'.format(file_name))
		torch.save(optimizer.state_dict(), 'model_outputs/{}/best_optimizer.pth'.format(file_name))
		best_val_loss = val_log
		print("=> saved best model as validation loss is lower than previous validation loss")
		trigger = 0
			
	#Save snapshot
	if torch.cuda.device_count() > 1:
		torch.save(model.module.state_dict(), 'model_outputs/{}/last_model.pth'.format(file_name))
	else:
		torch.save(model.state_dict(), 'model_outputs/{}/last_model.pth'.format(file_name))
	torch.save(optimizer.state_dict(), 'model_outputs/{}/last_optimizer.pth'.format(file_name))

	#early stopping
	
	if config.early_stopping >= 0 and trigger >= config.early_stopping:
		print("=> early stopping")
		break
			
	torch.cuda.empty_cache()

Creating directory called cnn_8_Adam_0.0001
--------------------
Configuration settings as follows:
epochs: 100
batch_size: 8
early_stopping: 40
num_workers: 8
optimizer: Adam
lr: 0.0001
weight_decay: 1e-05
--------------------
=> Creating model


Training:   0%|          | 0/100 [00:00<?, ?it/s]

labels únicos: tensor([0, 1, 2, 3], device='cuda:0')
min: 0
max: 3


labels únicos: tensor([0, 1, 3], device='cuda:0')
min: 0
max: 3
labels únicos: tensor([1, 2, 3], device='cuda:0')
min: 1
max: 3
labels únicos: tensor([0, 1, 2, 3], device='cuda:0')
min: 0
max: 3
labels únicos: tensor([1, 2, 3], device='cuda:0')
min: 1
max: 3
labels únicos: tensor([0, 1, 2, 3], device='cuda:0')
min: 0
max: 3
labels únicos: tensor([0, 1, 2, 3], device='cuda:0')
min: 0
max: 3


labels únicos: tensor([1, 2, 3], device='cuda:0')
min: 1
max: 3
labels únicos: tensor([0, 1, 2, 3], device='cuda:0')
min: 0
max: 3
labels únicos: tensor([0, 1, 2, 3], device='cuda:0')
min: 0
max: 3
labels únicos: tensor([0, 1, 3], device='cuda:0')
min: 0
max: 3
labels únicos: tensor([1, 3], device='cuda:0')
min: 1
max: 3


...

labels únicos: tensor([0, 1, 2, 3], device='cuda:0')
min: 0
max: 3
labels únicos: tensor([0, 2, 3], device='cuda:0')
min: 0
max: 3
labels únicos: tensor([0, 1, 2, 3], device='cuda:0')
min: 0
max: 3
labels únicos: tensor([1, 2, 3], device='cuda:0')
min: 1
max: 3
labels únicos: tensor([0, 1, 3], device='cuda:0')
min: 0
max: 3
labels únicos: tensor([0, 1, 2, 3], device='cuda:0')
min: 0
max: 3


Training: 511it [00:27, 18.91it/s]                             


labels únicos: tensor([0, 1, 3], device='cuda:0')
min: 0
max: 3
labels únicos: tensor([1, 3], device='cuda:0')
min: 1
max: 3
labels únicos: tensor([0, 3], device='cuda:0')
min: 0
max: 3


Validation: 127it [00:06, 19.83it/s]
Training:  48%|████▊     | 48/100 [26:15<28:53, 33.34s/it]

Training epoch [48/100], Training loss:0.7051, Validation loss:0.7083
=> saved best model as validation loss is lower than previous validation loss
